# La idea de la difusión: ensuciar una foto para aprender a limpiarla

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial — IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento Digital de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Ver con una foto real cómo se agrega ruido, cómo un modelo lo quita y qué tiene que ver eso con restaurar imágenes.

## Al terminar este cuaderno vas a poder:

1. Explicar con una imagen qué significa agregar y quitar ruido paso a paso.
2. Leer en un gráfico cuánto de la foto original queda en cada paso `t`.
3. Entender por qué el parámetro `strength` decide cuánto puede inventar un modelo al restaurar.

## Preparación

Esta unidad corre en **Google Colab** y lee las imágenes desde tu **Drive**. Antes de empezar, la carpeta `015_modelos_difusion` (con `datos/` adentro) tiene que estar en *Mi unidad*.

`diffusers` es la biblioteca de Hugging Face para trabajar con modelos de difusión. `torch` ya viene instalado en Colab.

In [ ]:
%pip install -q diffusers

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/015_modelos_difusion'  # carpeta de la unidad en tu Drive
DATOS = f'{CARPETA}/datos'  # imágenes de ejemplo

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'  # funciona sin GPU, pero más lento
print('Dispositivo:', dispositivo)

## 1. Una foto de 1909

Usamos una fotografía en color de Prokudin-Gorsky (1909, dominio público). Es una foto **real y nítida**: la vamos a ensuciar a propósito.

Los modelos no trabajan con valores de 0 a 255, sino de **-1 a 1**. La forma `(1, 3, alto, ancho)` significa: una imagen, tres canales (rojo, verde, azul), alto y ancho.

In [ ]:
foto = Image.open(f'{DATOS}/original_kirillov_1909.jpg').convert('RGB')
foto = foto.resize((400, 346))  # achicamos para que todo corra rápido

x0 = torch.tensor(np.array(foto), dtype=torch.float32) / 127.5 - 1  # de 0-255 a -1..1
x0 = x0.permute(2, 0, 1).unsqueeze(0)  # (alto, ancho, canal) -> (1, canal, alto, ancho)

print('Forma:', tuple(x0.shape))
print('Valor mínimo y máximo:', x0.min().item(), x0.max().item())
foto

## 2. Agregar ruido: el proceso hacia adelante

Un **planificador** (*scheduler*) define cuánto ruido se agrega en cada **paso** `t`. `DDPMScheduler` usa 1000 pasos: en `t = 0` la foto está intacta y en `t = 999` ya es casi ruido puro.

`add_noise(imagen, ruido, t)` mezcla la foto con el ruido que le pasamos. Usamos **siempre el mismo ruido** para que lo único que cambie entre imágenes sea `t`.

In [ ]:
from diffusers import DDPMScheduler

planificador = DDPMScheduler(num_train_timesteps=1000)  # 1000 pasos, como en el entrenamiento original

generador = torch.Generator().manual_seed(7)  # la semilla (seed) fija el ruido
ruido = torch.randn(x0.shape, generator=generador)  # ruido gaussiano del mismo tamaño que la foto

pasos = [0, 100, 250, 500, 750, 999]

figura, ejes = plt.subplots(1, len(pasos), figsize=(18, 3.4))
for numero in range(len(pasos)):
    t = torch.tensor([pasos[numero]])
    x_t = planificador.add_noise(x0, ruido, t)  # foto + ruido, según el paso t
    imagen = ((x_t[0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).numpy()  # de vuelta a 0..1 para mostrar
    ejes[numero].imshow(imagen)
    ejes[numero].set_title(f't = {pasos[numero]}')
    ejes[numero].axis('off')
plt.show()

## 3. Cuánto queda de la foto

En cada paso, la imagen ruidosa es una mezcla:

`x_t = (peso de la foto) · foto + (peso del ruido) · ruido`

El planificador guarda esos pesos. Cuando el peso de la foto baja, el modelo tiene **menos información real** y tiene que **suponer** más.

In [ ]:
alfa_acumulado = planificador.alphas_cumprod  # cuánto "sobrevive" de la foto hasta cada paso
peso_foto = alfa_acumulado.sqrt()
peso_ruido = (1 - alfa_acumulado).sqrt()

plt.figure(figsize=(7, 3.5))
plt.plot(peso_foto, label='peso de la foto')
plt.plot(peso_ruido, label='peso del ruido')
plt.xlabel('paso t')
plt.ylabel('peso en la mezcla')
plt.legend()
plt.show()

for t in pasos:
    print(f't = {t:>3}: foto {peso_foto[t]:.2f}   ruido {peso_ruido[t]:.2f}')

> **Para mirar.** Fijate dónde se cruzan las dos curvas. A partir de ese paso, hay más ruido que foto.

## 4. Quitar ruido: el modelo predice el ruido

El modelo (una red **UNet**) recibe una imagen ruidosa y el paso `t`, y **predice el ruido que contiene**. Si restamos ese ruido, damos un pasito hacia una imagen más limpia. Repetimos 1000 veces.

Usamos un modelo chico, `google/ddpm-cifar10-32`: genera imágenes de 32 × 32 píxeles y no entiende texto. Alcanza para **ver** el proceso. Empezamos desde ruido puro y no le damos ninguna foto.

In [ ]:
from diffusers import UNet2DModel

modelo = UNet2DModel.from_pretrained('google/ddpm-cifar10-32').to(dispositivo)  # la red que predice el ruido
planificador_inverso = DDPMScheduler.from_pretrained('google/ddpm-cifar10-32')  # reglas con las que se entrenó

millones = sum(p.numel() for p in modelo.parameters()) / 1e6
print(f'La red tiene {millones:.1f} millones de parámetros')

In [ ]:
torch.manual_seed(3)
x = torch.randn(1, 3, 32, 32).to(dispositivo)  # arrancamos de ruido puro

instantes = [999, 800, 600, 400, 200, 100, 50, 0]
capturas = []

planificador_inverso.set_timesteps(1000)
for t in planificador_inverso.timesteps:  # de 999 hacia 0
    with torch.no_grad():  # no estamos entrenando, solo usando el modelo
        ruido_predicho = modelo(x, t).sample  # el modelo adivina el ruido que hay en x
    x = planificador_inverso.step(ruido_predicho, t, x).prev_sample  # un paso hacia una imagen más limpia
    if int(t) in instantes:
        capturas.append((int(t), x.clone()))

figura, ejes = plt.subplots(1, len(capturas), figsize=(18, 2.8))
for numero in range(len(capturas)):
    t, imagen = capturas[numero]
    imagen = ((imagen[0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).cpu().numpy()
    ejes[numero].imshow(imagen, interpolation='nearest')
    ejes[numero].set_title(f'después de t = {t}')
    ejes[numero].axis('off')
plt.show()

> **Para mirar.** En ningún momento el modelo vio una foto de entrada: **inventó** una imagen plausible a partir del ruido. Cambiá `torch.manual_seed(3)` por otro número y vas a obtener otra imagen.

## 5. Qué tiene que ver esto con restaurar

Cuando restaurás una foto con un modelo de difusión (modo *image-to-image*), no empezás desde ruido puro: **le agregás ruido a tu foto hasta cierto paso y dejás que el modelo lo quite**.

El parámetro `strength` (de 0 a 1) decide hasta dónde se ensucia la foto:

In [ ]:
for strength in [0.1, 0.3, 0.5, 0.8, 1.0]:
    t = int(strength * 999)  # paso en el que arranca la limpieza
    print(f'strength = {strength:.1f}  ->  arranca en t = {t:>3}  |  peso de tu foto: {peso_foto[t]:.2f}')

- `strength` **bajo**: queda mucho de tu foto; el modelo corrige detalles y casi no inventa.
- `strength` **alto**: queda poco de tu foto; el modelo reconstruye casi todo y lo que ves es, en gran parte, **suposición suya**.

Esa es la pregunta de fondo del TPI: ¿qué parte de lo que ves es dato y qué parte es invento?

> **Para probar.** Volvé a la sección 2 y cambiá la lista `pasos` por los `t` que imprimió la tabla. Mirá cuánto de la foto sobrevive con `strength = 0.5`.

## Glosario

| Término | Qué significa acá |
|---|---|
| **Ruido gaussiano** | Valores al azar que se suman a cada píxel. |
| **Paso (`t`)** | Momento del proceso: 0 es la foto limpia, 999 es casi puro ruido. |
| **Planificador (*scheduler*)** | Define cuánto ruido se agrega o se quita en cada paso. |
| **UNet** | Red neuronal que, dada una imagen ruidosa y el paso `t`, predice el ruido. |
| **Seed (semilla)** | Número que fija el azar: misma semilla, mismo ruido. |
| **`strength`** | Cuánto se ensucia tu foto antes de restaurarla. |